# Compiling logical experiments from Q#

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we write logical quantum programs in Q# and run them through our upgraded qodec?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Declare top-layer `qodec` instructions as Q# intrinsic operations under `TargetProfile.Adaptive_RI`.
- Compile SPAM, two-block `cx_all`, and `syndrome` experiments to QIR with `qsharp.compile`.
- Execute compiled QIR through a `qodec` with `qdk.simulation.run_qir` across single and multiple rounds.

## Setting up

The cell below loads the distance-two `C4` protocol and initializes Q# with `TargetProfile.Adaptive_RI`, which is required for custom measurement operations.

In [ ]:
from qdk import TargetProfile, ec, qsharp
from qdk.simulation import run_qir
from _unit import make_upgraded_protocol

protocol = make_upgraded_protocol()
assert ec.audit(protocol).ok

qsharp.init(target_profile=TargetProfile.Adaptive_RI)

## Declaring qodec instructions as Q# intrinsics

Whether you author logical programs in Q# or OpenQASM 3, the underlying `qodec` and its verified gadgets stay unchanged—only the front-end compiler changes. When `run_qir(qir, qodec=protocol, ...)` executes a QIR program, each quantum call must match the `mnemonic` of an instruction in the `qodec`'s top layer. Arguments bind in a fixed order:
1. Qubit arguments bind to **block operands** (each Q# `Qubit` represents one encoded `C4` block, not one physical qubit).
2. Classical scalar arguments (`Double`, `Int`, `Bool`) bind in declaration order to **instruction parameters** (which gadgets forward into circuits via `parameter_bindings`).
3. Returned `Result` values receive **observable outcomes** first, followed by any returned **flags**.

When a qubit's first call has only `out` operands (like `prepare_z_all` or `prepare_x_all`), that call creates its block directly; otherwise, the runtime looks for a default `out`-only instruction with `action: [stabilize: Z_0]` and fails before running shots if none exists.

> **Warning — Current Q# intrinsic sharp edges (evolving interface):** Today, Q# `body intrinsic;` declarations are written by hand rather than auto-generated from a `qodec`, so several sharp edges apply:
> - **Exact signature matching:** Operation names, `Qubit` block counts, scalar parameter order, and `Result` tuple arity (`observables + returned flags`) must match the top-layer `InstructionSet` manually—mismatches surface only when `run_qir` lowers the compiled QIR.
> - **Profile and attribute requirements:** Any custom intrinsic returning one or more `Result` values requires `qsharp.init(target_profile=TargetProfile.Adaptive_RI)` and `@Measurement()`.
> - **Block-level handles:** One Q# `Qubit` always represents an entire encoded block (even when $k > 1$, as in `C4`), so individual logical qubits inside a multi-qubit block cannot be addressed as separate Q# `Qubit` values.
> - **Upcoming ergonomics:** Automatic stub generation and tighter compiler integration for `qodec` instruction sets are active areas of development and are expected to simplify (and potentially change) this boilerplate in future QDK releases.

For our upgraded `C4` protocol, `prepare_z_all` and `prepare_x_all` return one `Result` for the `reject` flag, while `measure_z_all` and `measure_x_all` return `(Result, Result)` for the two logical qubits:

In [ ]:
qsharp.eval("""
@Measurement()
operation prepare_z_all(block : Qubit) : Result { body intrinsic; }
@Measurement()
operation prepare_x_all(block : Qubit) : Result { body intrinsic; }
@Measurement()
operation measure_z_all(block : Qubit) : (Result, Result) { body intrinsic; }
@Measurement()
operation measure_x_all(block : Qubit) : (Result, Result) { body intrinsic; }
operation cx_all(control : Qubit, target : Qubit) : Unit { body intrinsic; }
operation syndrome(block : Qubit) : Unit { body intrinsic; }

operation TestSpamZ() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    let (left, right) = measure_z_all(block);
    ([reject], [left, right])
}

operation TestSyndromeZ() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    syndrome(block);
    let (left, right) = measure_z_all(block);
    ([reject], [left, right])
}

operation TestCnotX() : (Result[], Result[]) {
    use (ctl, tgt) = (Qubit(), Qubit());
    let r0 = prepare_x_all(ctl);
    let r1 = prepare_x_all(tgt);
    cx_all(ctl, tgt);
    let (c0, c1) = measure_x_all(ctl);
    let (t0, t1) = measure_x_all(tgt);
    ([r0, r1], [c0, c1, t0, t1])
}
""")

experiments = {
    "SPAM Z": str(qsharp.compile("TestSpamZ()")),
    "Syndrome Z": str(qsharp.compile("TestSyndromeZ()")),
    "CNOT X": str(qsharp.compile("TestCnotX()")),
}
for line in experiments["Syndrome Z"].splitlines():
    if "call" in line and "__quantum__" not in line:
        print("QIR call:", line.strip())

## Running noiseless and multi-round experiments

Notice how `qsharp.compile` preserves the logical calls (`@prepare_z_all`, `@syndrome`, `@measure_z_all`) inside the QIR. When we pass that QIR and `qodec=protocol` to `run_qir(..., type="stabilizer")`, the simulator lowers each logical call through its gadget, chains output stabilizer frames from one call to the next—even inside `for` loops—and decodes the measurements using the built-in syndrome decoder:

In [ ]:
for label, qir in experiments.items():
    shots = run_qir(qir, qodec=protocol, type="stabilizer", shots=10, seed=42)
    all_zero = all(
        all(b == qsharp.Result.Zero for b in flags + logical)
        for flags, logical in shots
    )
    assert all_zero
    print(f"{label:<12} -> first shot: {shots[0]}")

qsharp.eval("""
operation RunMemoryZ(rounds : Int) : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    for _ in 1..rounds {
        syndrome(block);
    }
    let (z0, z1) = measure_z_all(block);
    ([reject], [z0, z1])
}
""")
memory_4r = str(qsharp.compile("RunMemoryZ(4)"))
memory_shots = run_qir(memory_4r, qodec=protocol, type="stabilizer", shots=4, seed=7)
print("4-round memory sample:", memory_shots)

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does measure_z_all return two Result values in Q# even though its operand has type Qubit?</summary>

<div style="padding:0.1em 1em;">

In QIR lowered through a `qodec`, each `Qubit` handle represents one encoded `C4` block. Because `measure_z_all` observes both logical qubits (`z[0]` and `z[1]`, declared as `Observe(['Z_0', 'Z_1'])`), its `@Measurement()` signature returns a 2-tuple of `Result` values.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does on_shot_failure='discard' keep rows where the returned preparation flag is One?</summary>

<div style="padding:0.1em 1em;">

A flag triggers the runtime's `on_shot_failure` policy only when the logical program does not consume that flag. Because `prepare_z_all` was declared to return `Result`, Q# receives the flag explicitly so our benchmark script can count both attempted and accepted shots.

</div>

</details></div>

## Exercise: Execute a two-round Q# logical memory experiment

The starter code compiles the two-round logical $Z$-memory operation `Memory2R()`. Execute `10` noiseless shots on `protocol` with `run_qir(..., type="stabilizer")` and return the number of shots whose flag and logical results are all `Result.Zero`.

In [ ]:
from qdk import TargetProfile, qsharp
from qdk.simulation import run_qir
from _unit import exercise, make_upgraded_protocol


@exercise
def noiseless_two_round_memory_shots() -> int:
    protocol = make_upgraded_protocol()
    qsharp.init(target_profile=TargetProfile.Adaptive_RI)
    qsharp.eval('''
    @Measurement() operation prepare_z_all(b : Qubit) : Result { body intrinsic; }
    @Measurement() operation measure_z_all(b : Qubit) : (Result, Result) { body intrinsic; }
    operation syndrome(b : Qubit) : Unit { body intrinsic; }
    operation Memory2R() : (Result[], Result[]) {
        use b = Qubit();
        let r = prepare_z_all(b);
        syndrome(b);
        syndrome(b);
        let (z0, z1) = measure_z_all(b);
        ([r], [z0, z1])
    }
    ''')
    qir = str(qsharp.compile('Memory2R()'))
    # Run 10 noiseless stabilizer shots and count those with only zero results.
    return ...

**Hint**

Call `shots = run_qir(qir, qodec=protocol, type='stabilizer', shots=10, seed=42)`. Each shot is a `(flags, logical)` pair. Count the shot only if every result in both lists is `qsharp.Result.Zero`.

In [ ]:
from qdk import TargetProfile, qsharp
from qdk.simulation import run_qir
from _unit import exercise, make_upgraded_protocol


@exercise
def noiseless_two_round_memory_shots() -> int:
    protocol = make_upgraded_protocol()
    qsharp.init(target_profile=TargetProfile.Adaptive_RI)
    qsharp.eval('''
    @Measurement() operation prepare_z_all(b : Qubit) : Result { body intrinsic; }
    @Measurement() operation measure_z_all(b : Qubit) : (Result, Result) { body intrinsic; }
    operation syndrome(b : Qubit) : Unit { body intrinsic; }
    operation Memory2R() : (Result[], Result[]) {
        use b = Qubit();
        let r = prepare_z_all(b);
        syndrome(b);
        syndrome(b);
        let (z0, z1) = measure_z_all(b);
        ([r], [z0, z1])
    }
    ''')
    qir = str(qsharp.compile('Memory2R()'))
    shots = run_qir(qir, qodec=protocol, type='stabilizer', shots=10, seed=42)
    return sum(
        all(x == qsharp.Result.Zero for x in flags + logical)
        for flags, logical in shots
    )

`10`. `run_qir` lowers each `syndrome(b)` call in sequence and chains the block's output stabilizer signs into the next round's input checks automatically.

## Further reading

- [Run and decode logical programs ↗](https://learn.microsoft.com/en-us/azure/quantum/run-decode-logical-programs)
- [Call qodec instructions from Q# (sample notebook) ↗](https://github.com/microsoft/qdk/blob/main/samples/notebooks/qdk_ec/qodec_instructions_from_qsharp.ipynb)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../06-fault-tolerant-gadgets/fault_tolerant_gadgets.workbook.ipynb" title="Previous unit: Upgrading Gadgets for Fault Tolerance" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Upgrading Gadgets for Fault Tolerance</span></span></a><a href="../08-openqasm-integration/openqasm_integration.workbook.ipynb" title="Next unit: Calling Qodec Instructions from OpenQASM" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Calling Qodec Instructions from OpenQASM</span></span><span aria-hidden="true">&#8250;</span></a></div>